Lade zunächst die Datei


In [1]:
annotated_by = {}
with open("annotatedBy.txt", encoding="utf-8") as f:
  for line in f:
    line = line.rstrip("\n")
    if not line:
      continue
    parts = line.split("\t")
    if len(parts) >= 2:
      key = parts[0]
      value = "\t".join(parts[1:])
      annotated_by[key] = [item.strip() for item in value.split(", ") if item.strip()]
print(len(annotated_by))

3322


Lösche alle Dokumente, die von zwei oder mehr Personen annotiert wurden.


In [2]:
annotated_done = {key: value for key, value in annotated_by.items() if len(value) > 1}
annotated_by = {key: value for key, value in annotated_by.items() if len(value) <= 1}
annotated_by0 = {key: value for key, value in annotated_by.items() if len(value) == 0}
annotated_by1 = {key: value for key, value in annotated_by.items() if len(value) == 1}

print(len(annotated_done))
print(len(annotated_by))
print(len(annotated_by0))
print(len(annotated_by1))

1187
2135
1533
602


Liste mit Nutzern an die die Dokumente verteilt werden.


In [3]:
users = set(['Konstantina Karakechagia', 'Lea Wassmer', 'Lara Baum'])

Skript zur Verteilung (möglichst gleichmäßig)


In [4]:
import itertools
import os

user_list = sorted(users)

task_counts = {user: 0 for user in user_list}
stage1_counts = {user: 0 for user in user_list}
stage2_counts = {user: 0 for user in user_list}

script_lines = [
  "#!/bin/sh"
]
for user in user_list:
  script_lines.append(f'mkdir -p "output/{user}";')

for key, annotators in annotated_by1.items():
  candidates = [user for user in user_list if user not in annotators]
  if not candidates:
    raise RuntimeError(f"No candidate available for {key}")
  chosen = min(candidates, key=lambda user: (task_counts[user], user))
  task_counts[chosen] += 1
  stage1_counts[chosen] += 1
  script_lines.append(f'cp "input/{key}.json" "output/{chosen}/";')

for key in annotated_by0:
  pair = min(
    itertools.combinations(user_list, 2),
    key=lambda pair: (
      task_counts[pair[0]] + task_counts[pair[1]],
      max(task_counts[pair[0]], task_counts[pair[1]]),
      pair,
    )
  )
  for user in pair:
    task_counts[user] += 1
    stage2_counts[user] += 1
    script_lines.append(f'cp "input/{key}.json" "output/{user}/";')

script_path = "distribute_to_users.sh"
with open(script_path, "w", encoding="utf-8") as f:
  f.write("\n".join(script_lines) + "\n")

os.chmod(script_path, 0o755)

print(f"Shell script written to: {script_path}")
print(f"Stage 1 files (annotated_by1): {len(annotated_by1)}")
print(f"Stage 2 files (annotated_by0): {len(annotated_by0)}")
print("Final assignment counts:")
for user in user_list:
  print(f"- {user}: total {task_counts[user]} (stage1 {stage1_counts[user]}, stage2 {stage2_counts[user]})")

Shell script written to: distribute_to_users.sh
Stage 1 files (annotated_by1): 602
Stage 2 files (annotated_by0): 1533
Final assignment counts:
- Konstantina Karakechagia: total 1223 (stage1 242, stage2 981)
- Lara Baum: total 1223 (stage1 241, stage2 982)
- Lea Wassmer: total 1222 (stage1 119, stage2 1103)


Folgender Befehl für das Zusammenführen von Dateien in neuem Ordner


In [5]:
find /opt/data -type f -exec cp -n {} /opt/data2/ \;

SyntaxError: invalid syntax (246186703.py, line 1)